# Data Processing

<br><br>

## 0. Imports

In [141]:
import numpy as np
import pandas as pd

<br><br>

# 1. Load Data

In [142]:
diabetes_raw = pd.read_csv('data/diabetes/diabetic_data.csv')

<br><br>

## 2. Clean Data

In [143]:
# Drop unknown gender
diabetes_clean = diabetes_raw[diabetes_raw['gender'] != 'Unknown/Invalid']

diabetes_clean['race'] = diabetes_clean['race'].fillna('Unknown')
diabetes_clean['payer_code'] = diabetes_clean['payer_code'].fillna('Unknown')
diabetes_clean['medical_specialty'] = diabetes_clean['medical_specialty'].fillna('Unknown')
diabetes_clean['max_glu_serum'] = diabetes_clean['max_glu_serum'].fillna('None')
diabetes_clean['A1Cresult'] = diabetes_clean['A1Cresult'].fillna('None')

# Drop high-missing columns
diabetes_clean = diabetes_clean.drop(columns = 'weight')

# Drop patients who cannot be readmitted
diabetes_clean = diabetes_clean[
    ~diabetes_clean['discharge_disposition_id'].isin({11, 13, 14, 19, 20, 21})
]

# Clean columns
diabetes_clean = diabetes_clean.replace({'?': np.nan})
diabetes_clean['readmitted'] = diabetes_clean['readmitted'].replace({'NO': 0,'>30': 0,'<30': 1}).astype(int)
diabetes_clean['diabetesMed'] = diabetes_clean['diabetesMed'].replace({'No': 0, 'Yes': 1}).astype(int)
diabetes_clean['change'] = diabetes_clean['change'].replace({'No': 0, 'Ch': 1}).astype(int)
diabetes_clean['gender'] = diabetes_clean['gender'].replace({'Male': 1, 'Female': 0}).astype(int)

# Keep only the first record for each patient_nbr
diabetes_clean = diabetes_clean.sort_values('encounter_id')
diabetes_clean = diabetes_clean.groupby('patient_nbr').first().reset_index()

# Drop identifiers
diabetes_clean = diabetes_clean.drop(columns=['encounter_id', 'patient_nbr'])

In [144]:
diabetes_clean.columns

Index(['race', 'gender', 'age', 'admission_type_id',
       'discharge_disposition_id', 'admission_source_id', 'time_in_hospital',
       'payer_code', 'medical_specialty', 'num_lab_procedures',
       'num_procedures', 'num_medications', 'number_outpatient',
       'number_emergency', 'number_inpatient', 'diag_1', 'diag_2', 'diag_3',
       'number_diagnoses', 'max_glu_serum', 'A1Cresult', 'metformin',
       'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride',
       'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide',
       'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone',
       'tolazamide', 'examide', 'citoglipton', 'insulin',
       'glyburide-metformin', 'glipizide-metformin',
       'glimepiride-pioglitazone', 'metformin-rosiglitazone',
       'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted'],
      dtype='str')

<br><br>

## 3. Process Data

In [145]:
diabetes_processed = diabetes_clean.copy()

### 3.1. Transform Age

In [146]:
# Map age groups to numeric values
diabetes_processed['age'] = diabetes_processed['age'].map({
    '[0-10)': 5,
    '[10-20)': 15,
    '[20-30)': 25,
    '[30-40)': 35,
    '[40-50)': 45,
    '[50-60)': 55,
    '[60-70)': 65,
    '[70-80)': 75,
    '[80-90)': 85,
    '[90-100)': 95
})

### 3.2. Transform Medicin

In [147]:
medication_cols = [
    'metformin',
    'repaglinide',
    'nateglinide',
    'chlorpropamide',
    'glimepiride',
    'acetohexamide',
    'glipizide',
    'glyburide',
    'tolbutamide',
    'pioglitazone',
    'rosiglitazone',
    'acarbose',
    'miglitol',
    'troglitazone',
    'tolazamide',
    'examide',
    'citoglipton',
    'insulin',
    'glyburide-metformin',
    'glipizide-metformin',
    'glimepiride-pioglitazone',
    'metformin-rosiglitazone',
    'metformin-pioglitazone'
]

for col in medication_cols:
    diabetes_processed[col] = diabetes_processed[col].apply(
        lambda x: 0 if x == 'No' else 1
    )

In [148]:
rare_meds = [
    'acarbose',
    'chlorpropamide',
    'tolazamide',
    'miglitol',
    'tolbutamide',
    'glipizide-metformin',
    'troglitazone',
    'metformin-rosiglitazone',
    'acetohexamide',
    'glimepiride-pioglitazone',
    'metformin-pioglitazone',
    'examide',
    'citoglipton'
]

diabetes_processed["other_meds"] = (
    diabetes_processed[rare_meds].sum(axis=1) > 0
).astype(int)

diabetes_processed = diabetes_processed.drop(columns=rare_meds)

### 3.3. Transform Diagnoses

In [149]:
def categorize_diagnosis(diag):
    if pd.isna(diag):
        return "Unknown"

    diag = str(diag)

    # V-codes
    if diag.startswith("V"):
        return "Supplementary"

    # E-codes
    if diag.startswith("E"):
        return "External Causes"

    try:
        diag_num = float(diag)

        # 001–139.9
        if 1 <= diag_num < 140:
            return "Infectious and Parasitic Diseases"

        # 140–239.9
        elif 140 <= diag_num < 240:
            return "Neoplasms"

        # 240–279.9
        elif 240 <= diag_num < 280:
            return "Endocrine, Nutritional and Metabolic Diseases"

        # 280–289.9
        elif 280 <= diag_num < 290:
            return "Blood and Blood-Forming Diseases"

        # 290–319
        elif 290 <= diag_num < 320:
            return "Mental Disorders"

        # 320–389.9
        elif 320 <= diag_num < 390:
            return "Nervous System and Sense Organs"

        # 390–459.9
        elif 390 <= diag_num < 460:
            return "Circulatory System Diseases"

        # 460–519.9
        elif 460 <= diag_num < 520:
            return "Respiratory System Diseases"

        # 520–579.9
        elif 520 <= diag_num < 580:
            return "Digestive System Diseases"

        # 580–629.9
        elif 580 <= diag_num < 630:
            return "Genitourinary System Diseases"

        # 630–676.9
        elif 630 <= diag_num < 677:
            return "Pregnancy and Childbirth Complications"

        # 680–709.9
        elif 680 <= diag_num < 710:
            return "Skin and Subcutaneous Tissue Diseases"

        # 710–739.9
        elif 710 <= diag_num < 740:
            return "Musculoskeletal and Connective Tissue Diseases"

        # 740–759.9
        elif 740 <= diag_num < 760:
            return "Congenital Anomalies"

        # 760–779.9
        elif 760 <= diag_num < 780:
            return "Perinatal Conditions"

        # 780–799.9
        elif 780 <= diag_num < 800:
            return "Symptoms, Signs and Ill-Defined Conditions"

        # 800–999.9
        elif 800 <= diag_num < 1000:
            return "Injury and Poisoning"

        else:
            return "Other"

    except:
        return "Other"
    
diabetes_processed["diag_1_group"] = diabetes_processed["diag_1"].apply(categorize_diagnosis)
diabetes_processed["diag_2_group"] = diabetes_processed["diag_2"].apply(categorize_diagnosis)
diabetes_processed["diag_3_group"] = diabetes_processed["diag_3"].apply(categorize_diagnosis)

# List all diagnosis group columns
diag_group_cols = [
    "diag_1_group",
    "diag_2_group",
    "diag_3_group"
]

# Get all unique disease categories
all_categories = pd.unique(
    diabetes_processed[diag_group_cols].values.ravel()
)

# Create binary columns
for category in all_categories:
    diabetes_processed[category] = (
        (diabetes_processed["diag_1_group"] == category) |
        (diabetes_processed["diag_2_group"] == category) |
        (diabetes_processed["diag_3_group"] == category)
    ).astype(int)

diabetes_processed = diabetes_processed.drop(columns=[
    "diag_1",
    "diag_2",
    "diag_3",
    "diag_1_group",
    "diag_2_group",
    "diag_3_group",
])

### 3.4. Combine Visit Counts

In [150]:
diabetes_processed['service_utilization'] = (
    diabetes_processed['number_outpatient'] + 
    diabetes_processed['number_emergency'] + 
    diabetes_processed['number_inpatient']
)

# Drop the individual counts
diabetes_processed = diabetes_processed.drop(columns=['number_outpatient', 'number_emergency', 'number_inpatient'])

In [151]:
diabetes_processed.info()

<class 'pandas.DataFrame'>
RangeIndex: 69987 entries, 0 to 69986
Data columns (total 49 columns):
 #   Column                                          Non-Null Count  Dtype
---  ------                                          --------------  -----
 0   race                                            68166 non-null  str  
 1   gender                                          69987 non-null  int64
 2   age                                             69987 non-null  int64
 3   admission_type_id                               69987 non-null  int64
 4   discharge_disposition_id                        69987 non-null  int64
 5   admission_source_id                             69987 non-null  int64
 6   time_in_hospital                                69987 non-null  int64
 7   payer_code                                      40591 non-null  str  
 8   medical_specialty                               37934 non-null  str  
 9   num_lab_procedures                              69987 non-null  int64
 1

### 3.5. One-Hot Encode Remaining Categorical Columns

In [152]:
# Reduce cardinality for medical_specialty
top_specialties = diabetes_processed['medical_specialty'].value_counts().nlargest(10).index
diabetes_processed['medical_specialty'] = diabetes_processed['medical_specialty'].apply(
    lambda x: x if x in top_specialties else 'Other'
)

# Encode
categorical_cols = ['race', 'payer_code', 'medical_specialty', 'max_glu_serum', 'A1Cresult']
diabetes_processed = pd.get_dummies(
    diabetes_processed, 
    columns=categorical_cols, 
    drop_first=True,
    dtype=int
)

In [153]:
def bin_admission_type(val):
    if val in [1, 2, 7]: return 'Emergency'
    if val == 3: return 'Elective'
    return 'Other'

def bin_discharge_disposition(val):
    if val == 1: return 'Home'
    if val in [3, 6, 8]: return 'Home_Health_SNF'
    if val in [2, 4, 5, 22, 23, 24, 27, 28, 29, 30]: return 'Transferred_Medical'
    return 'Other'

def bin_admission_source(val):
    if val == 7: return 'Emergency_Room'
    if val in [1, 2, 3]: return 'Referral'
    if val in [4, 5, 6, 10, 18, 22, 25, 26]: return 'Transfer'
    return 'Other'

# Apply transformations
diabetes_processed['admission_type'] = diabetes_processed['admission_type_id'].apply(bin_admission_type)
diabetes_processed['discharge_disposition'] = diabetes_processed['discharge_disposition_id'].apply(bin_discharge_disposition)
diabetes_processed['admission_source'] = diabetes_processed['admission_source_id'].apply(bin_admission_source)

# Drop original numerical ID columns
diabetes_processed = diabetes_processed.drop(columns=['admission_type_id', 'discharge_disposition_id', 'admission_source_id'])

# One-hot encode new categorical columns
categorical_to_encode = ['admission_type', 'discharge_disposition', 'admission_source']
diabetes_processed = pd.get_dummies(diabetes_processed, columns=categorical_to_encode, drop_first=True, dtype=int)

In [154]:
diabetes_processed.info()

<class 'pandas.DataFrame'>
RangeIndex: 69987 entries, 0 to 69986
Data columns (total 85 columns):
 #   Column                                          Non-Null Count  Dtype
---  ------                                          --------------  -----
 0   gender                                          69987 non-null  int64
 1   age                                             69987 non-null  int64
 2   time_in_hospital                                69987 non-null  int64
 3   num_lab_procedures                              69987 non-null  int64
 4   num_procedures                                  69987 non-null  int64
 5   num_medications                                 69987 non-null  int64
 6   number_diagnoses                                69987 non-null  int64
 7   metformin                                       69987 non-null  int64
 8   repaglinide                                     69987 non-null  int64
 9   nateglinide                                     69987 non-null  int64
 1

<br><br>

## 4. Save Processed Data

In [155]:
diabetes_processed.to_csv('data/diabetes_processed.csv', index = False)